In [4]:
!pip install airfrans

  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.6 MB 14.9 MB/s eta 0:00:01
   -------------------------------------- - 2.5/2.6 MB 26.6 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 18.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/219.0 kB ? eta -:--:--
   --------------------------------------- 219.0/219.0 kB 13.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/81.3 MB ? eta -:--:--
    --------------------------------------- 1.0/81.3 MB 33.4 MB/s eta 0:00:03
    --------------------------------------- 1.0/81.3 MB 33.4 MB/s eta 0:00:03
    --------------------------------------- 1.0/81.3 MB 33.4 MB/s eta 0:00:03
    --------------------------------------- 1.8/81.3 MB 10.1 MB/s eta 0:00:08
   - -------------------------------------- 2.1/81.3 MB 11.1 MB/s eta 0:00:08
   - ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires packaging<24,>=16.8, but you have packaging 25.0 which is incompatible.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 6.33.6 which is incompatible.
streamlit 1.32.0 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.


In [5]:
import airfrans as af
directory_name='/kaggle/input/airfrans/Dataset'
dataset_list, dataset_name = af.dataset.load(root = directory_name, task = 'scarce', train = True)

C:\Users\lucap\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/airfrans/Dataset\\manifest.json'

In [2]:
import pandas as pd

# 1. Agarramos la primera simulación de la lista (un solo perfil alar)
simulacion = dataset_list[0]



NameError: name 'dataset_list' is not defined

In [6]:
!pip install airfrans pandas
import airfrans as af
import pandas as pd

print("Descargando directo de la fuente...")
# Esto lo baja y lo extrae sin depender de la interfaz web
af.dataset.download(root='.', file_name='Dataset', unzip=True, OpenFOAM=False)

print("Cargando los datos...")
dataset_list, dataset_name = af.dataset.load(root='./Dataset', task='scarce', train=True)
simulacion = dataset_list[0]

print("Armando el CSV 2D...")
df = pd.DataFrame({
    'x': simulacion.position[:, 0],
    'y': simulacion.position[:, 1],
    'u': simulacion.velocity[:, 0],
    'v': simulacion.velocity[:, 1],
    'wall_dist': simulacion.wall_distance[:, 0]
})

df.to_csv('ala_2d_crudo.csv', index=False)
print("¡Listo para Julia!")

Descargando directo de la fuente...


URLError: <urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>

In [7]:
pip install pyvista

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pyvista in c:\users\lucap\anaconda3\lib\site-packages (0.48.4)



In [8]:
import pyvista as pv
import pandas as pd
import numpy as np

# 1. Cargamos el archivo de malla 3D/2D
malla = pv.read('airFoil2D_SST_33.763_-1.179_4.54_5.165_6.778_internal.vtu')

# 2. Extraemos las coordenadas espaciales
puntos = malla.points
df = pd.DataFrame({
    'x': puntos[:, 0],
    'y': puntos[:, 1]
})

# 3. Extraemos las velocidades (U)
# En los archivos OpenFOAM/VTK, la velocidad suele llamarse 'U'
if 'U' in malla.point_data:
    velocidades = malla.point_data['U']
    df['u'] = velocidades[:, 0]
    df['v'] = velocidades[:, 1]
else:
    print("¡Ojo! No encontré la variable 'U'. Las variables disponibles son:", malla.point_data.keys())

# 4. Condición de borde: Distancia a la pared (nut o wallDistance)
# Buscamos la variable que indique la distancia a la superficie del ala
if 'wallDistance' in malla.point_data:
    df['wall_dist'] = malla.point_data['wallDistance']
elif 'nut' in malla.point_data: # A veces usan la viscosidad turbulenta cinemática nula en la pared
    df['nut'] = malla.point_data['nut']
    
# 5. Exportar el bendito CSV
df.to_csv('ala_2d_crudo.csv', index=False)
print("¡Éxito total! Archivo ala_2d_crudo.csv listo para Julia.")

¡Éxito total! Archivo ala_2d_crudo.csv listo para Julia.


In [9]:
import pyvista as pv
import pandas as pd

# 1. Cargamos la malla
malla = pv.read('airFoil2D_SST_33.763_-1.179_4.54_5.165_6.778_internal.vtu') # Reemplazá con tu nombre de archivo

# Diccionario vacío donde vamos a ir metiendo todas las columnas
datos_dicc = {}

# 2. Guardamos las coordenadas espaciales (x, y, z)
puntos = malla.points
datos_dicc['x'] = puntos[:, 0]
datos_dicc['y'] = puntos[:, 1]
# Omitimos 'z' porque al ser 2D será todo 0, pero podrías agregarla.

# 3. Extraemos TODAS las variables dinámicamente
for nombre_var in malla.point_data.keys():
    arreglo = malla.point_data[nombre_var]
    
    # Si la variable es escalar (1 dimensión, ej: presión 'p', 'nut')
    if len(arreglo.shape) == 1:
        datos_dicc[nombre_var] = arreglo
        
    # Si la variable es un vector (2 o más dimensiones, ej: velocidad 'U')
    elif len(arreglo.shape) == 2:
        # La separamos en sus componentes X, Y, Z
        ejes = ['x', 'y', 'z']
        for i in range(arreglo.shape[1]):
            datos_dicc[f"{nombre_var}_{ejes[i]}"] = arreglo[:, i]

# 4. Convertimos el diccionario a un DataFrame de Pandas
df_completo = pd.DataFrame(datos_dicc)

# 5. Lo guardamos en un nuevo CSV
df_completo.to_csv('ala_2d_completo.csv', index=False)

print("¡Archivo ala_2d_completo.csv generado exitosamente!")
print("-" * 40)
print("Las columnas que encontramos en este archivo son:")
print(df_completo.columns.tolist())

¡Archivo ala_2d_completo.csv generado exitosamente!
----------------------------------------
Las columnas que encontramos en este archivo son:
['x', 'y', 'nut', 'p', 'U_x', 'U_y', 'U_z', 'implicit_distance', 'vtkOriginalPointIds']
